In [1]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
huggingface_dataset_name="knkarthick/dialogsum"

dataset=load_dataset(huggingface_dataset_name)

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [5]:
example_indices=[40,200]

dash_line='-'.join("" for x in range(100))
for i,index in enumerate(example_indices):
    print(dash_line)
    print("Example",i+1)
    print(dash_line)
    print("INPUT DIALOGUE:")
    print(dataset['test'][index]['dialogue'])
    print(dash_line)
    print("BASELINE HUMAN SUMMARY:")
    print(dataset['test'][index]['summary'])
    print(dash_line)
    print()

---------------------------------------------------------------------------------------------------
Example 1
---------------------------------------------------------------------------------------------------
INPUT DIALOGUE:
#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
---------------------------------------------------------------------------------------------------

---------------------------------------------------------------------------------------------------
Exam

In [6]:
model_name='google/flan-t5-base'
model=AutoModelForSeq2SeqLM.from_pretrained(model_name)

W1016 22:44:06.344000 32528 venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [7]:
tokenizer=AutoTokenizer.from_pretrained(model_name,use_fast=True)

In [8]:
sentence="what time is it,Tom?"

sentence_encoded=tokenizer(sentence, return_tensors="pt")

sentence_decoded=tokenizer.decode(
    sentence_encoded['input_ids'][0],
    skip_special_tokens=True
)

print("Encoded Sentence:")
print(sentence_encoded["input_ids"][0])
print("Decoded Sentence:")
print(sentence_decoded)

Encoded Sentence:
tensor([ 125,   97,   19,   34,    6, 3696,   51,   58,    1])
Decoded Sentence:
what time is it,Tom?


In [9]:
tokenizer.decode(model.generate(sentence_encoded['input_ids'],max_new_tokens=50)[0],skip_special_tokens=True)

'4:00 PM'

## NO PROMPTING

In [10]:
for i,index in enumerate(example_indices):
    dialogue=dataset['test'][index]['dialogue']
    summary=dataset['test'][index]['summary']

    inputs=tokenizer(dialogue,return_tensors="pt")
    output=tokenizer.decode(
        model.generate(
            inputs['input_ids'],
            max_new_tokens=50
        )[0],
        skip_special_tokens=True
    )

    print(dash_line)
    print("Example",i+1)
    print(dash_line)
    print(f"INPUT PROMPT:\n{dialogue}")
    print(f"BASELINE HUMAN SUMMARY:\n{summary}")
    print(dash_line)
    print(f"MODEL GENERATION - WITHOUT PROMPT ENGINEERING:\n{output}\n")

---------------------------------------------------------------------------------------------------
Example 1
---------------------------------------------------------------------------------------------------
INPUT PROMPT:
#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
BASELINE HUMAN SUMMARY:
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
---------------------------------------------------------------------------------------------------
MODEL GENERATION - WITHOUT PROMPT ENGINEERING:
Person1: It's ten to nine.

---------------------------------------------------------------------------------------------------
Example 2
----------------------

## ZERO SHOT PROMPTING

In [11]:
for i,index in enumerate(example_indices):
    dialogue=dataset['test'][index]['dialogue']
    summary=dataset['test'][index]['summary']
    prompt=f"""
    Summarize the following conversation
    {dialogue}

    Summary :
    """
    inputs=tokenizer(prompt,return_tensors="pt")
    output=tokenizer.decode(
        model.generate(
            inputs['input_ids'],
            max_new_tokens=50
        )[0],
        skip_special_tokens=True
    )

    print(dash_line)
    print("Example",i+1)
    print(dash_line)
    print(f"INPUT PROMPT:\n{dialogue}")
    print(f"BASELINE HUMAN SUMMARY:\n{summary}")
    print(dash_line)
    print(f"MODEL GENERATION - ZERO SHOT:\n{output}\n")

---------------------------------------------------------------------------------------------------
Example 1
---------------------------------------------------------------------------------------------------
INPUT PROMPT:
#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
BASELINE HUMAN SUMMARY:
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
---------------------------------------------------------------------------------------------------
MODEL GENERATION - ZERO SHOT:
The train is about to leave.

---------------------------------------------------------------------------------------------------
Example 2
-------------------------------------

In [12]:
for i,index in enumerate(example_indices):
    dialogue=dataset['test'][index]['dialogue']
    summary=dataset['test'][index]['summary']
    prompt=f"""
    Dialogue:
    {dialogue}

    What is going on?
    """
    inputs=tokenizer(prompt,return_tensors="pt")
    output=tokenizer.decode(
        model.generate(
            inputs['input_ids'],
            max_new_tokens=50
        )[0],
        skip_special_tokens=True
    )

    print(dash_line)
    print("Example",i+1)
    print(dash_line)
    print(f"INPUT PROMPT:\n{dialogue}")
    print(f"BASELINE HUMAN SUMMARY:\n{summary}")
    print(dash_line)
    print(f"MODEL GENERATION - ZERO SHOT:\n{output}\n")

---------------------------------------------------------------------------------------------------
Example 1
---------------------------------------------------------------------------------------------------
INPUT PROMPT:
#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
BASELINE HUMAN SUMMARY:
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
---------------------------------------------------------------------------------------------------
MODEL GENERATION - ZERO SHOT:
Tom is late for the train.

---------------------------------------------------------------------------------------------------
Example 2
---------------------------------------

## ONE SHOT PROMPTING

In [13]:
def make_prompt(example_indices_full,example_indices_to_summarize):
    prompt=""
    for index in example_indices_full:
        dialogue=dataset['test'][index]['dialogue']
        summary=dataset['test'][index]['summary']

        prompt+=f"""
        Dialogue:

        {dialogue}

        What was going on?

        {summary}
        """
        dialogue=dataset["test"][example_indices_to_summarize]['dialogue']

    prompt+=f"""
    Dialogue:

    {dialogue}

    What was going on?
    """
    return prompt

In [14]:
example_indices_full=[40]
example_index_to_summarize=200

In [15]:
one_shot_prompt=make_prompt(example_indices_full,example_index_to_summarize)
print(one_shot_prompt)


        Dialogue:

        #Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.

        What was going on?

        #Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
        
    Dialogue:

    #Person1#: Have you considered upgrading your system?
#Person2#: Yes, but I'm not sure what exactly I would need.
#Person1#: You could consider adding a painting program to your software. It would allow you to make up your own flyers and banners for advertising.
#Person2#: That would be a definite bonus.
#Person1#: You might also want to upgrade your hardware because it is pretty outdated now.
#Person2#: How can we do that?
#Person1#: You'd probably need a f

In [16]:
summary=dataset['test'][example_index_to_summarize]['summary']

input=tokenizer(one_shot_prompt,return_tensors="pt")
output=tokenizer.decode(
    model.generate(
        input['input_ids'],
        max_new_tokens=50
    )[0],
    skip_special_tokens=True
)

print(dash_line)
print(f"BASELINE HUMAN SUMMARY:\n{summary}\n")
print(dash_line)
print(f"MODEL GENERATION - ONE SHOT:\n{output}")

---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.

---------------------------------------------------------------------------------------------------
MODEL GENERATION - ONE SHOT:
#Person1 wants to upgrade his system. #Person2 wants to add a painting program to his software. #Person1 wants to add a CD-ROM drive.


## FEW SHOT PROMPTING

In [17]:
example_indices_full=[40,80,120]
example_index_to_summarize=200

few_shot_prompt =make_prompt(example_indices_full,example_index_to_summarize)

print(few_shot_prompt)


        Dialogue:

        #Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.

        What was going on?

        #Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
        
        Dialogue:

        #Person1#: May, do you mind helping me prepare for the picnic?
#Person2#: Sure. Have you checked the weather report?
#Person1#: Yes. It says it will be sunny all day. No sign of rain at all. This is your father's favorite sausage. Sandwiches for you and Daniel.
#Person2#: No, thanks Mom. I'd like some toast and chicken wings.
#Person1#: Okay. Please take some fruit salad and crackers for me.
#Person2#: Done. Oh, don't forget to take napkins disposabl

In [18]:
summary=dataset['test'][example_index_to_summarize]['summary']

input=tokenizer(few_shot_prompt,return_tensors="pt")
output=tokenizer.decode(
    model.generate(
        input['input_ids'],
        max_new_tokens=50
    )[0],
    skip_special_tokens=True
)

print(dash_line)
print(f"BASELINE HUMAN SUMMARY:\n{summary}\n")
print(dash_line)
print(f"MODEL GENERATION - ONE SHOT:\n{output}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (818 > 512). Running this sequence through the model will result in indexing errors


---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.

---------------------------------------------------------------------------------------------------
MODEL GENERATION - ONE SHOT:
#Person1 wants to upgrade his system. #Person2 wants to add a painting program to his software. #Person1 wants to upgrade his hardware.


## TRYING PARAMETERS

In [19]:
generation_config=GenerationConfig(temperature=1.0,do_sample=True,max_new_tokens=50,top_k=5,top_p=0.5)

In [20]:
summary=dataset['test'][example_index_to_summarize]['summary']

input=tokenizer(few_shot_prompt,return_tensors="pt")
output=tokenizer.decode(
    model.generate(
        input['input_ids'],
        generation_config=generation_config
    )[0],
    skip_special_tokens=True
)

print(dash_line)
print(f"BASELINE HUMAN SUMMARY:\n{summary}\n")
print(dash_line)
print(f"MODEL GENERATION - ONE SHOT:\n{output}")

---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.

---------------------------------------------------------------------------------------------------
MODEL GENERATION - ONE SHOT:
#Person1 recommends upgrading their system and hardware.
